# SCI Eligible Cohort — Base Logic

**Data:** NSCISC public-use database, version 2021ARPublic. Form I (`f1`) has one row per person (initial acute care and rehab stay). Form II (`f2`) has one row per person per follow-up year (post-injury years 1, 5, 10, 15, …).
**Rules:** SCI inclusion criteria, slide 28, Fall 2026 Week 01 Introductory Presentation.
**Codes:** every code decision comes from the NSCISC *Data Dictionary (Public, AR2021)* and the *Quick Reference Guide (2021)*. Page numbers are noted next to each variable.

## Inclusion rules

| Step | Slide 28 rule | Variable(s) | Person is included if… |
|---|---|---|---|
| 0 | Rehab at a Model Systems hospital + part of SCI Model Systems data collection | Form I | They have a Form I record. Form I is only completed for people enrolled through a Model Systems hospital. |
| 1 | Has a discharge location | `APResDis` (Form I) | Discharge location is a real place (codes 1–6, 8, 9, 10). Excluded: 7 = Deceased, 99 = Unknown. |
| 2 | Has a current living location | `BPlcRes` (Form II) | At least one follow-up year has a known place of residence. |
| 3 | Has rehospitalization status since last data collection | `BRhspNbr` (Form II) | At least one follow-up year has a known rehospitalization status. |
| 4 | 2+ time points of Social Determinants of Health data | SDOH variables (Form II) | At least 2 follow-up years have a known value in at least one SDOH variable. |
| 5 | 2+ time points of Functional Ability data | Functional variables (Form II) | At least 2 follow-up years have a known value in at least one functional variable. |

## Definitions
- **Time point** = one Form II follow-up year (`BYear`) where the person was followed, i.e. not lost to follow-up (`BFolUpCt` ≠ 5).
- **Known value** = a real answer. Unknown, declined, don't know and not applicable codes are treated as missing (Quick Reference Guide, note 4). The exact codes for each variable are in the codebook below.
- **Variables for each category.** The slide gives examples for each category, and each example is covered:
  - *SDOH:* employment (`BPrLvlSt`), healthcare access (`BPrimPay`), housing status (`BPlcRes`), plus marital status, education, family income and transportation.
  - *Functional Ability:* self-care (`BSFSCRS`, SCI-FI Self-Care) and PHQ (`BBPHQMDS`, `BBPHQSDS`), plus CHART Physical Independence (`BCHPITot`).

## 1. Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path(".")                     # <-- folder containing the NSCISC CSV files
f1 = pd.read_csv(DATA_DIR / "f1_public_2021.csv", low_memory=False)
f2 = pd.read_csv(DATA_DIR / "f2_public_2021.csv", low_memory=False)

# Structure checks (Quick Reference Guide, note 1)
assert f1["UniID"].is_unique, "Form I must have one row per person (UniID)"
assert not f2.duplicated(["UniID", "BYear"]).any(), "Form II must have one row per UniID + BYear"
print(f"Form I : {len(f1):,} people")
print(f"Form II: {len(f2):,} follow-up records")

## 2. Codebook (from the Data Dictionary)

For each variable, `valid` lists the codes that are real answers, and `missing` lists the codes meaning unknown, declined, don't know or not applicable. Blank cells are always missing.

If the data contains a code that isn't in the codebook, the notebook stops, so no code is treated as missing or valid without a documented decision.

In [ ]:
# Discharge location -- Form I #16, Place of Residence (Data Dictionary p. 30-31)
# 1 Private residence, 2 Hospital, 3 Nursing home, 4 Group living, 5 Correctional,
# 6 Hotel/motel, 7 Deceased, 8 Other (unclassified), 9 Homeless, 10 Assisted living, 99 Unknown
DISCHARGE_VALID = {1, 2, 3, 4, 5, 6, 8, 9, 10}      # 7 (Deceased) and 99 (Unknown) are not a discharge location

CODEBOOK = {
    # ---- Living location & rehospitalization ----
    # Form II #10 Place of Residence (p. 243): 1-6, 8, 9, 10 = residence types; 99 = Unknown
    "BPlcRes":  {"valid": {1, 2, 3, 4, 5, 6, 8, 9, 10}, "missing": {99}},
    # Form II #79 Number of Rehospitalizations (p. 312): 0 = none, 1-7 = count,
    # 8 = rehospitalized, number unknown (status is known = Yes); 9 = Unknown
    "BRhspNbr": {"valid": set(range(0, 9)), "missing": {9}},

    # ---- Social Determinants of Health ----
    # Form II #14 Marital Status (p. 245): 1-7 = status (7 = living with partner); 9 = Unknown
    "BMarStat": {"valid": set(range(1, 8)), "missing": {9}},
    # Form II #16 Education Level (p. 248): 1-7 = level, 8 = Other; 9 = Unknown
    "BEducLvl": {"valid": set(range(1, 9)), "missing": {9}},
    # Form II #83 Occupational Status -- EMPLOYMENT (p. 319): 1-10 = status (7 = Unemployed); 99 = Unknown
    "BPrLvlSt": {"valid": set(range(1, 11)), "missing": {99}},
    # Form II #86 Family Income (p. 324): 1-4 = income bracket; 6 = Doesn't know, 7 = Declined, 9 = Unknown
    "BFmIncLv": {"valid": {1, 2, 3, 4}, "missing": {6, 7, 9}},
    # Form II #81 Primary Payer -- HEALTHCARE ACCESS (p. 315): 1-9 = payer type; 77 = Declined/doesn't know, 99 = Unknown
    "BPrimPay": {"valid": set(range(1, 10)), "missing": {77, 99}},
    # Form II #124 Primary Mode of Transportation (p. 358): 0-7 = mode (7 = Walk); 9 = Unknown
    "BMdTrnsp": {"valid": set(range(0, 8)), "missing": {9}},

    # ---- Functional Ability ----
    # Form II #132 SCI-FI Self-Care Raw Score (p. 366): valid raw score; 99 = score imputed from
    # partially answered items (assessment WAS done, so it counts as data); blank = not done
    "BSFSCRS":  {"valid": set(range(0, 100)), "missing": set()},
    # Form II #114 CHART Physical Independence Total (p. 347): 0-100 = score; 999 = Unknown
    "BCHPITot": {"valid": set(range(0, 101)), "missing": {999}},
    # Form II #48 PHQ Major Depressive Syndrome (p. 268): 0-2 = result; 9 = Unknown/declined.
    # 7 is not listed in the dictionary but appears in the data -> treated as Declined (Quick Ref. note 4)
    "BBPHQMDS": {"valid": {0, 1, 2}, "missing": {7, 9}},
    # Form II #49 PHQ Severity of Depression (p. 270): 0-27 = score; 99 = Unknown/declined.
    # 77 is not listed in the dictionary but appears in the data -> treated as Declined (Quick Ref. note 4)
    "BBPHQSDS": {"valid": set(range(0, 28)), "missing": {77, 99}},
}

SDOH_VARS = ["BPlcRes", "BMarStat", "BEducLvl", "BPrLvlSt", "BFmIncLv", "BPrimPay", "BMdTrnsp"]
FUNC_VARS = ["BSFSCRS", "BCHPITot", "BBPHQMDS", "BBPHQSDS"]

## 3. Prepare Form II
1. Keep only follow-up years where the person was followed: drop records coded lost to follow-up (`BFolUpCt` = 5; Data Dictionary p. 227).
2. Check that every code in the data is in the codebook.
3. Set missing codes to blank (NaN), so a non-blank value always means a known value.

In [ ]:
# 1. Followed (not lost) follow-up years only
f2_followed = f2[pd.to_numeric(f2["BFolUpCt"], errors="coerce") != 5].copy()
print(f"Form II records: {len(f2):,} total -> {len(f2_followed):,} followed (not lost)")

# 2. Every code present must be listed in the codebook
problems = []
for var, book in CODEBOOK.items():
    values = pd.to_numeric(f2_followed[var], errors="coerce").dropna()
    unlisted = sorted(set(values) - book["valid"] - book["missing"])
    if unlisted:
        problems.append(f"{var}: {unlisted}")
assert not problems, "Codes not in the codebook (add them with a documented reason):\n" + "\n".join(problems)
print("All codes in the data are documented in the codebook.")

# 3. Missing codes -> NaN
for var, book in CODEBOOK.items():
    f2_followed[var] = pd.to_numeric(f2_followed[var], errors="coerce")
    f2_followed.loc[f2_followed[var].isin(book["missing"]), var] = np.nan

## 4. Apply the inclusion rules

In [ ]:
funnel = []   # (step, number of people remaining)

# Step 0 -- Model Systems rehab + data collection: everyone with a Form I record
eligible = set(f1["UniID"])
funnel.append(("0. Form I record (Model Systems rehab)", len(eligible)))

In [ ]:
# Step 1 -- Has a discharge location
discharge = pd.to_numeric(f1["APResDis"], errors="coerce")
has_discharge_location = set(f1.loc[discharge.isin(DISCHARGE_VALID), "UniID"])

eligible &= has_discharge_location
funnel.append(("1. Has a discharge location", len(eligible)))

In [ ]:
# Step 2 -- Has a current living location (known in at least one follow-up year)
has_living_location = set(f2_followed.loc[f2_followed["BPlcRes"].notna(), "UniID"])

eligible &= has_living_location
funnel.append(("2. Has a living location", len(eligible)))

In [ ]:
# Step 3 -- Has rehospitalization status (known in at least one follow-up year)
has_rehosp_status = set(f2_followed.loc[f2_followed["BRhspNbr"].notna(), "UniID"])

eligible &= has_rehosp_status
funnel.append(("3. Has rehospitalization status", len(eligible)))

In [ ]:
# Helper: number of follow-up years in which at least one of `variables` has a known value
def n_time_points(variables):
    year_has_data = f2_followed[variables].notna().any(axis=1)
    return f2_followed.loc[year_has_data].groupby("UniID")["BYear"].nunique()

# Step 4 -- 2+ time points of SDOH data
sdoh_time_points = n_time_points(SDOH_VARS)
eligible &= set(sdoh_time_points[sdoh_time_points >= 2].index)
funnel.append(("4. 2+ time points of SDOH data", len(eligible)))

In [ ]:
# Step 5 -- 2+ time points of Functional Ability data
func_time_points = n_time_points(FUNC_VARS)
eligible &= set(func_time_points[func_time_points >= 2].index)
funnel.append(("5. 2+ time points of Functional Ability data", len(eligible)))

## 5. Cohort funnel

In [ ]:
funnel_df = pd.DataFrame(funnel, columns=["Step", "People"])
funnel_df["Removed"] = (funnel_df["People"].shift(1) - funnel_df["People"]).astype("Int64")
funnel_df["% of Form I"] = (100 * funnel_df["People"] / funnel_df["People"].iloc[0]).round(1)
display(funnel_df)
print(f"Eligible cohort: N = {len(eligible):,}")

## 6. Save the cohort
Every analysis should start from this list of IDs so all pairs use the same cohort.

In [ ]:
cohort_ids = pd.Series(sorted(eligible), name="UniID")
cohort_ids.to_csv("eligible_cohort_ids.csv", index=False)

f1_cohort = f1[f1["UniID"].isin(eligible)].copy()
f2_cohort = f2[f2["UniID"].isin(eligible)].copy()     # all Form II rows for eligible people
print(f"Saved eligible_cohort_ids.csv  ({len(cohort_ids):,} people, {len(f2_cohort):,} Form II rows)")

## Notes for the team

**Decisions made from the Data Dictionary**
- **Discharge code 8 (Other, unclassified) is kept.** The dictionary lists it as a real place of residence. Only 7 (Deceased) and 99 (Unknown) mean there's no discharge location.
- **Rehospitalization code 8** ("rehospitalized, number unknown") is kept. The status is known (yes), only the count is missing.
- **SCI-FI Self-Care 99** means the score was imputed because some items weren't answered. The assessment was done, so it counts as data for eligibility. Don't use 99 as a numeric score in analysis.
- **Codes in the data that the dictionary doesn't list:** `BBPHQMDS` = 7 and `BBPHQSDS` = 77. They're treated as *Declined* (Quick Reference Guide, note 4: "Declined is 7s").

**How the rules are read**
- **Living location and rehospitalization status** must be known in *at least one* follow-up year.
- **A time point counts for a category** if *any one* variable in that category is known that year. SDOH and functional time points are counted separately, so they don't need to be the same years.

**Note on the old N = 13,041:** this notebook follows the Data Dictionary exactly, so its N may differ slightly from earlier notebooks, which used different missing-value codes.